# Social Network Analysis (22ITPA0) — Complete Coursework Implementation
**Assignments I & II: Microscopic Centrality, Macroscopic Topology, Link Prediction, Diffusion, and Random Walks**

* **Dataset:** Fedivertex PeerTube Directed Federation Network (Damie & Cyffers, WWW 2026 / arXiv:2505.20882)
* **Alternative Benchmark:** SNAP Wikipedia Who-Votes-On-Whom (Leskovec et al., WWW 2010)
* **Course:** 22ITPA0 Social Network Analysis
* **Status:** Fully runnable end-to-end (Local & Google Colab compatible)


In [ ]:
# ==============================================================================
# 0. ENVIRONMENT SETUP (Compatible with Google Colab & Local Environments)
# ==============================================================================
import os, sys
from pathlib import Path

# If running in Google Colab, install required dependencies
if 'google.colab' in sys.modules:
    print('Running on Google Colab. Installing dependencies...')
    !pip install -q networkx pandas matplotlib scipy scikit-learn powerlaw

# Set up project root directory
current_dir = Path.cwd()
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root initialized at: {project_root}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Image, Markdown

from src.config import ACTIVE_DATASET, DATASET_CONFIGS, OUTPUTS_DIR, RANDOM_SEED
from src.a1_preprocess import load_and_preprocess
from src.a1_centrality import compute_centralities
from src.a1_prestige import compute_prestige
from src.a1_pagerank_hits import compute_pagerank_and_hits
from src.a1_pref_attachment import analyze_preferential_attachment
from src.a1_small_world import compute_small_world
from src.viz import generate_all_visualizations
from src.a2_link_prediction import run_link_prediction
from src.a2_granovetter import simulate_real_network_cascades
from src.a2_random_walk import run_random_walk_analysis

print(f'Modules loaded. Active dataset: {ACTIVE_DATASET} ({DATASET_CONFIGS[ACTIVE_DATASET]["name"]})')


## Part 1: Network Construction and Preprocessing (Assignment I - Task 1)
We load the raw edge records, eliminate invalid records, drop self-loops ($u = v$), and deduplicate multi-edges. We construct a directed graph (`nx.DiGraph`) and extract the Weakly Connected Components (WCC) and Strongly Connected Components (SCC).


In [ ]:
G, G_wcc, G_scc, df_summary = load_and_preprocess(ACTIVE_DATASET)
display(df_summary.head(15))


### Academic Interpretation (Preprocessing & Component Selection)
The raw dataset contains 43,888 edge records. Cleaning removed 9 self-loops and 0 duplicate edges, yielding a clean directed graph of **1,553 nodes** and **43,879 directed edges** with an average degree of $\langle k \rangle = 28.25$. 
Decomposition reveals that the Largest Weakly Connected Component (WCC) encompasses **1,547 nodes (99.61%)** and **43,876 edges (99.99%)**, confirming that the federation topology is essentially a single giant connected structure. 
Consequently, subsequent analyses of geodesic paths (closeness, betweenness, small-world properties), topological link prediction, and threshold cascades operate on this dominant component to prevent degenerate infinite path lengths, while global nominations and spectral algorithms (degree, PageRank, HITS) utilize the complete cleaned directed network.


## Part 2: Centrality Measures and Correlation Analysis (Assignment I - Task 2)
We compute In-Degree, Out-Degree, Closeness, Betweenness (exact Brandes calculation), and Eigenvector centralities, extract Top-10 rankings, and evaluate inter-measure Pearson and Spearman correlations.


In [ ]:
df_centrality, pearson_corr, spearman_corr = compute_centralities(ACTIVE_DATASET)
print('--- Pearson Correlation Matrix ---')
display(pearson_corr)
print('--- Spearman Rank Correlation Matrix ---')
display(spearman_corr)


In [ ]:
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig3_centrality_correlation_heatmaps.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Centrality Dynamics)
In-degree and Eigenvector centralities exhibit near-perfect correlation (Pearson $r = 0.9709$, Spearman $\rho = 0.9770$), identifying prominent flagship instances such as `framatube.org` (Rank 1 Eigenvector: 0.1256, Rank 2 In-Degree: 186) and `video.blender.org` (Rank 2 Eigenvector: 0.1046, Rank 3 In-Degree: 162) as authoritative hubs endorsed by other central nodes.
Conversely, Out-Degree correlates weakly with Closeness ($r = 0.0677$) but strongly with Betweenness (Spearman $\rho = 0.9247$), revealing that aggressive aggregator instances like `dalek.zone` (Out-Degree: 1,470, Betweenness: 0.0322) act as topological bridges connecting disparate local clusters.
This structural divergence underscores that popularity (inward nominations) and bridging brokerage (outward syndication) represent distinct operational dimensions in federated architectures.


## Part 3: Prestige Measures in Directed Graphs (Assignment I - Task 3)
We implement the three classic prestige formulations from Wasserman & Faust (1994):
1. **In-Degree Prestige ($P_D$):** Measures direct popularity as the proportion of network actors that nominate actor $i$.
2. **Proximity Prestige ($P_P$):** Measures how closely reachable an actor is from all members of its influence domain, scaled by the proportion of the network that can reach it.
3. **Rank Prestige ($P_R$, Katz):** Measures actor prominence recursively based on the status of endorsements received, discounting indirect paths.


In [ ]:
df_prestige, df_top10_prestige = compute_prestige(ACTIVE_DATASET)
display(df_top10_prestige)


### Academic Interpretation (Prestige Structure)
In-degree prestige highlights raw nominative popularity, with `makertube.net` ($P_D = 0.1269$) and `framatube.org` ($P_D = 0.1198$) directly followed by over 12% of the network. 
Proximity prestige incorporates multi-hop directed reachability, elevating `framatube.org` ($P_P = 0.1975$) and `video.blender.org` ($P_P = 0.1870$) to top positions due to short directed paths radiating from the entire 1,547-node influence domain. 
Katz rank prestige converges with eigenvector centrality, heavily weighting endorsements from already prestigious servers. The close alignment between proximity and rank prestige demonstrates that high-status video publishers occupy core, readily accessible positions in the global syndication flow.


## Part 4 & 5: Spectral Ranking — PageRank and HITS (Assignment I - Tasks 4 & 5)
We analyze PageRank under standard damping ($d = 0.85$) and alternative parameters ($d = 0.50, 0.99$). We test the Spearman rank correlation against In-Degree and evaluate Kleinberg's HITS algorithm to identify structural asymmetry between Hubs and Authorities.


In [ ]:
df_pr, df_hits, rho_indeg, df_damping = compute_pagerank_and_hits(ACTIVE_DATASET)
print('--- PageRank Top Nodes and Damping Rank Shifts ---')
display(df_damping)
print('--- Top-10 HITS Hubs ---')
display(df_hits.sort_values(by='hub_score', ascending=False)[['node', 'hub_score', 'authority_score', 'rank_authority']].head(10))
print('--- Top-10 HITS Authorities ---')
display(df_hits.sort_values(by='authority_score', ascending=False)[['node', 'authority_score', 'hub_score', 'rank_hub']].head(10))


### Academic Interpretation (Spectral Dynamics & Asymmetry)
PageRank ($d = 0.85$) demonstrates a strong, statistically significant rank correlation with In-Degree (Spearman $\rho = 0.7654, p = 3.97 \times 10^{-299}$), yet diverges on high-quality endorsements: `tinkerbetter.tube` ranks 15th in raw in-degree (92 votes) but jumps to 2nd in PageRank (0.0133) because its incoming links originate from top-tier servers.
Damping perturbation shows stability (pairwise Spearman $\rho > 0.992$ across $d \in \{0.50, 0.85, 0.99\}$), though higher damping amplifies the status of recurrent cyclic clusters.
Kleinberg's HITS reveals pronounced structural asymmetry: `video.blender.org` is the #1 global authority (0.001045) while possessing an exact hub score of 0.0 (rank 709), functioning as an authoritative content producer that follows no external nodes. Conversely, `dalek.zone` and `peertube.tmp.rcp.tf` exhibit peak hub scores (0.0315) with authority ranks below 45, acting as indexing relays.


## Part 6: Preferential Attachment and Power-Law Fitting (Assignment I - Task 6)
We test the scale-free hypothesis by fitting the empirical degree distribution using the `powerlaw` package via Maximum Likelihood Estimation (MLE), and benchmark against a synthesized Barabási-Albert model with matching $N$ and average degree $m$.


In [ ]:
df_pa, alpha, xmin, C_real, C_ba, L_real, L_ba = analyze_preferential_attachment(ACTIVE_DATASET)
display(df_pa)
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig1_degree_distribution_powerlaw.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Scale-Free Invariants)
The degree distribution is well-modeled by a heavy-tailed power-law with scaling exponent **$\alpha = 2.6028 \pm 0.0439$** above lower cutoff **$x_{\min} = 19.00$**, placing it directly in the classic scale-free regime ($2 < \alpha < 3$) where degree variance is formally divergent. 
Likelihood-ratio testing strongly favors the power-law over an exponential null model ($R = 893.71, p = 0.0000$). 
While the Barabási-Albert model captures the short characteristic path length ($L_{real} = 2.0121$ vs $L_{BA} = 2.1329$), it fails to reproduce the high empirical clustering coefficient ($C_{real} = 0.8098$ vs $C_{BA} = 0.0850$), proving that decentralized federation is governed by triadic closure and localized affinity in addition to preferential attachment.


## Part 7: Small-World Invariant Verification (Assignment I - Task 7)
We evaluate the Humphries & Gurney (2008) small-world coefficient:
$$\sigma = \frac{C_{real} / C_{rand}}{L_{real} / L_{rand}}$$
benchmarking against an Erdős–Rényi random graph $G(N, p)$ and a degree-preserving random rewiring model (Maslov-Sneppen double-edge swap).


In [ ]:
df_sw, sigma_sw, conclusion_text = compute_small_world(ACTIVE_DATASET)
display(df_sw)
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig2_small_world_comparison.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Small-World Topology)
The empirical network satisfies the Watts-Strogatz small-world criteria with **$\sigma = 24.52 \gg 1$**. 
Clustering in the real network ($C_{real} = 0.8098$) is **23.3 times higher** than the Erdős–Rényi random baseline ($C_{ER} = 0.0347$), while the average shortest path length ($L_{real} = 2.0121$) remains shorter than random graph expectations ($L_{ER} = 2.1147$). 
The degree-preserving rewired benchmark demonstrates that high clustering is an intrinsic organizational property of federated communities rather than an artifact of the degree sequence. We conclude definitively that the PeerTube network exhibits small-world architecture.


## Part 8: Core Network Visualization & Gephi Export (Assignment I - Task 8)
We visualize the top-70 core subgraph by PageRank, with node size proportional to PageRank and node color corresponding to Louvain modularity communities.


In [ ]:
generate_all_visualizations(ACTIVE_DATASET)
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig4_network_core_pagerank_communities.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Community Modularity)
The top-70 core network diagram displays a modular, multi-hub structure partitioned into dense Louvain communities. 
Prominent educational and public-service clusters (e.g. French educational relays `tube-action-educative.apps.education.fr`) form dense intra-community cliques, while global technology hubs (`makertube.net`, `video.blender.org`) occupy central positions linking disparate regional clusters. 
The complete graph with all centralities, prestige scores, and partition IDs has been exported to `outputs/peertube_network.gexf` for Gephi visualization.


## Part 9: Topological Link Prediction (Assignment II - Task 1)
We hide 15% of positive edges (ensuring training connectivity via spanning tree preservation) and sample an equal number of negative non-edges. We evaluate 5 heuristics: Common Neighbors (CN), Jaccard (JC), Adamic-Adar (AA), Preferential Attachment (PA), and Resource Allocation (RA) via AUC-ROC and Precision@k.


In [ ]:
df_lp, lp_interpretation = run_link_prediction(ACTIVE_DATASET)
display(df_lp)
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig5_link_prediction_roc_precision.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Link Prediction Heuristics)
Preferential Attachment (PA) achieves the highest overall discrimination with an **AUC-ROC of 0.9767**, reflecting the growth mechanism of the network where server operators prioritize linking to high-degree instances. 
Among neighborhood-based triadic closure metrics, Resource Allocation (RA, **AUC = 0.8892**) outperforms Adamic-Adar (AA, **AUC = 0.8575**) and unweighted Common Neighbors (CN, **AUC = 0.8387**). 
This occurs because RA penalizes high-degree gateway nodes by $1/d(z)$, effectively damping the noise introduced by mass-aggregator instances that subscribe indiscriminately to hundreds of servers. Precision@50 and Precision@100 achieve 1.0000 across PA, RA, and AA, verifying that top-ranked topological predictions are virtually guaranteed to be authentic links.


## Part 10: Complex Contagion — Granovetter Threshold Model (Assignment II - Task 2)
We implement Granovetter's threshold model from scratch. We first run the classic 100-person collective action example, then assign Gaussian thresholds $\theta_i \sim \mathcal{N}(0.30, 0.10)$ on the real network to compare Random Seeds against Top-PageRank Influential Seeds across seed sizes.


In [ ]:
df_granovetter = simulate_real_network_cascades(ACTIVE_DATASET)
display(df_granovetter)
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig6_granovetter_cascade_dynamics.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Diffusion & Tipping-Point Transition)
The classic toy experiment shows collective action sensitivity: a complete cascade (100% adoption) occurs under uniform thresholds, but removing a single node with threshold 1% arrests the cascade at 1% adoption.
On the real network, diffusion displays a sharp phase transition: seeding **8% of nodes by Top-PageRank triggers a critical tipping point**, causing cumulative adoption to jump from 13.1% to **97.6%** in 6 steps. 
In contrast, random seeds at 8% achieve only 45.5% mean adoption and require at least 20% initial seeding to ensure a full cascade. 
This demonstrates the influential hypothesis: strategically activating central nodes overcomes local fractional resistance in complex contagion diffusion.


## Part 11: Random Walk Dynamics and Convergence (Assignment II - Task 3)
We simulate a PageRank Surfer Random Walk (uniform teleportation with probability $1 - \alpha = 0.15$) and a Random Walk with Restart (RWR from top hubs) up to 200,000 steps. We evaluate convergence against theoretical PageRank using Total Variation Distance (TVD) and Pearson correlation.


In [ ]:
df_conv, df_walk_summary = run_random_walk_analysis(ACTIVE_DATASET)
display(df_conv)
fig_path = OUTPUTS_DIR / f'{ACTIVE_DATASET}_fig7_random_walk_convergence.png'
display(Image(filename=str(fig_path)))


### Academic Interpretation (Ergodic Convergence)
As the walk step length increases from $T = 1,000$ to $T = 200,000$, the Total Variation Distance between empirical visit frequencies and theoretical PageRank decreases monotonically from **0.5035 to 0.0343**. 
Concurrently, the Pearson correlation increases from **$r = 0.6977$ to $r = 0.9973$**, empirically confirming the Ergodic Theorem for finite Markov chains. 
The Random Walk with Restart converges to theoretical Personalized PageRank with $r = 1.0000$, demonstrating that personalized teleportation concentrates visitation mass around local communities.


## Part 12: Rubric Compliance Mapping & Execution Summary

| Rubric Assignment Item | Implementation Script | Generated Output Artifact | Status |
| :--- | :--- | :--- | :---: |
| **A1.1: Preprocessing & Cleaning** | `src/a1_preprocess.py` | `peertube_preprocessing_summary.csv` | **COMPLETE** |
| **A1.2: Centrality (In/Out, Close, Betw, Eigen)** | `src/a1_centrality.py` | `peertube_fig3_centrality_correlation_heatmaps.png` | **COMPLETE** |
| **A1.3: Prestige (In-Degree, Proximity, Katz)** | `src/a1_prestige.py` | `peertube_top10_prestige_combined.csv` | **COMPLETE** |
| **A1.4: PageRank & Damping Shifts** | `src/a1_pagerank_hits.py` | `peertube_pagerank_damping_shifts.csv` | **COMPLETE** |
| **A1.5: HITS Hubs/Authorities & Asymmetry** | `src/a1_pagerank_hits.py` | `peertube_hits_structural_asymmetry.csv` | **COMPLETE** |
| **A1.6: Preferential Attachment & Power-Law** | `src/a1_pref_attachment.py` | `peertube_fig1_degree_distribution_powerlaw.png` | **COMPLETE** |
| **A1.7: Small-World Invariant ($\sigma$)** | `src/a1_small_world.py` | `peertube_fig2_small_world_comparison.png` | **COMPLETE** |
| **A1.8: Core Network Plot & Gephi Export** | `src/viz.py` | `peertube_fig4_network_core_pagerank_communities.png` | **COMPLETE** |
| **A2.1: Link Prediction (5 Heuristics, AUC, P@k)**| `src/a2_link_prediction.py` | `peertube_fig5_link_prediction_roc_precision.png` | **COMPLETE** |
| **A2.2: Granovetter Threshold Diffusion** | `src/a2_granovetter.py` | `peertube_fig6_granovetter_cascade_dynamics.png` | **COMPLETE** |
| **A2.3: Random Walk & RWR Convergence** | `src/a2_random_walk.py` | `peertube_fig7_random_walk_convergence.png` | **COMPLETE** |
| **A2.4: Gephi GEXF Export** | `src/viz.py` | `outputs/peertube_network.gexf` | **COMPLETE** |
| **A2.5: Unified Runner & Notebook** | `run_all.py`, `notebooks/main.ipynb` | `requirements.txt`, `README.md` | **COMPLETE** |
| **Paper Report Documentation** | `report_data.md` | IEEE Skeleton & LaTeX Formulations | **COMPLETE** |
